# CPA Calculation & Budget Reallocation Flags

**Goal:** Turn Shapley-attributed conversion credit (from `Models.ipynb`) into an
actionable **cost-per-attributed-conversion (CPA)** metric per brand-channel pair, by
combining it with *actual* ad spend derived from real event counts — then flag
channels that look like candidates for defunding or frequency capping.

**Why "actual" spend rather than budgeted spend:** Campaign budgets in
`campaign_spend.csv` represent planned allocation, not what was actually consumed.
Real spend depends on real event volume (CPC channels are billed per click, CPM
channels per 1,000 impressions), so actual spend is derived directly from the cleaned
touchpoint counts rather than read off the budget sheet.

**Inputs:** `per_brand_attribution.csv` (Shapley shares from `Models.ipynb`),
`touchpoints_clean_v3.csv` (cleaned event log), `campaign_spend.csv` (pricing model +
rate card per brand-channel)
**Outputs:** `spend_actual.csv`, `cpa_final.csv`

In [2]:
import pandas as pd
import numpy as np

In [3]:
COL_BRAND = "brand_id"
COL_CH = "channel"

## Step 1 — Compute actual spend per brand-channel

Counts real `Click` and `Impression` events from the cleaned touchpoints per
(brand, channel), then prices them against each campaign's `Cost_Rate_INR` and
`Pricing_Model`:

- **CPC channels** → `actual_spend = n_clicks × cost_rate_inr`
- **CPM channels** → `actual_spend = n_impressions × cost_rate_inr / 1000`

This produces a ground-truth spend figure per brand-channel, independent of whatever
was originally budgeted.

In [7]:
def compute_actual_spend(touchpoints_df: pd.DataFrame, spend_df: pd.DataFrame) -> pd.DataFrame:
    """
    Count real Click/Impression events per (brand, channel) from the
    cleaned touchpoints, then price them using each campaign's
    Cost_Rate_INR + Pricing_Model from campaign_spend.csv.
    """
    tp = touchpoints_df.copy()
    tp["brand_id"] = tp["campaign_id"].str.extract(r"_(B\d+)_")
    tp["channel"] = tp["channel"].str.strip().str.title()

    event_counts = (
        tp.groupby(["brand_id", "channel", "event_type"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    for col in ["Impression", "Click"]:
        if col not in event_counts.columns:
            event_counts[col] = 0
    event_counts = event_counts.rename(columns={
        "Impression": "n_impressions", "Click": "n_clicks"
    })[[COL_BRAND, COL_CH, "n_impressions", "n_clicks"]]

    spend = spend_df.rename(columns={
        "Brand_ID": COL_BRAND, "Channel": COL_CH,
        "Pricing_Model": "pricing_model", "Cost_Rate_INR": "cost_rate_inr",
        "Total_Budget_Allocated": "budget_allocated",
    })[[COL_BRAND, COL_CH, "pricing_model", "cost_rate_inr", "budget_allocated"]]
    spend[COL_CH] = spend[COL_CH].str.strip().str.title()

    merged = spend.merge(event_counts, on=[COL_BRAND, COL_CH], how="left")
    merged[["n_impressions", "n_clicks"]] = merged[["n_impressions", "n_clicks"]].fillna(0)

    merged["actual_spend"] = np.where(
        merged["pricing_model"] == "CPC",
        merged["n_clicks"] * merged["cost_rate_inr"],
        merged["n_impressions"] * merged["cost_rate_inr"] / 1000,
    )
    merged["actual_spend"] = merged["actual_spend"].round(2)

    return merged


## Step 2 — Compute CPA using Shapley-attributed conversions

Rather than crediting a channel with the *raw count* of conversions it touched
(which would double-count conversions across every channel in a multi-touch
journey), conversions are fractionally attributed using the Shapley share computed
in `Models.ipynb`:

```
attributed_conversions = (shapley_pct / 100) × n_converted
cpa = actual_spend / attributed_conversions
```

This is the key step that ties the attribution model to a dollar (₹) figure — a
channel with a small Shapley share gets proportionally little credit for
conversions, so its effective CPA reflects how efficiently it's *actually*
contributing, not just how much was spent on it.

In [10]:
def compute_cpa(per_brand_attribution: pd.DataFrame, spend_actual: pd.DataFrame) -> pd.DataFrame:
    """
    attributed_conversions = (shapley_pct / 100) * n_converted   [per brand-channel]
    CPA = actual_spend / attributed_conversions
    """
    merged = per_brand_attribution.merge(spend_actual, on=[COL_BRAND, COL_CH], how="left")

    merged["attributed_conversions"] = round(
        (merged["shapley_pct"] / 100) * merged["n_converted"], 1)

    merged["cpa"] = np.where(
        merged["attributed_conversions"] > 0,
        round(merged["actual_spend"] / merged["attributed_conversions"], 0),
        np.nan)

    return merged

## Step 3 — Flag defund and frequency-cap candidates

Two heuristic flags, both relative *within each brand* (since CPA scale varies a lot
brand to brand):

- **Defund candidate:** CPA is at least `2×` the brand's cheapest channel CPA, **and**
  that channel holds less than `15%` of Shapley credit. The logic: a channel that's
  expensive *and* clearly not earning its keep (low credit share) is a defund
  candidate. A channel that's expensive but holds high Shapley credit is deliberately
  **not** flagged — that's "expensive but working," a budget-reallocation
  conversation, not a cut-it conversation.
- **Frequency-cap candidate:** impression volume is in the top quartile within the
  brand, but Shapley credit is below the brand's median. This catches channels
  burning a lot of impressions for comparatively little attributed credit — a
  classic frequency-capping (reduce exposure, don't necessarily defund) signal,
  distinct from the defund case above.

The threshold values (`2.0×`, `15%`, `75th percentile`) are tunable parameters with
sensible defaults baked into the function signature, not hardcoded magic numbers.

In [13]:
def flag_defund_and_frequency_cap(cpa_df: pd.DataFrame,
                                   defund_multiplier: float = 2.0,
                                   defund_credit_threshold: float = 15.0,
                                   high_impression_threshold_pctile: float = 0.75
                                   ) -> pd.DataFrame:
    """
    DEFUND flag:
      A channel within a brand is a defund candidate if its CPA is more
      than `defund_multiplier`x the brand's cheapest channel CPA, AND it
      holds less than `defund_credit_threshold`% of Shapley credit.
      (High CPA + low credit = paying a premium for a channel that isn't
      earning its keep. High CPA + high credit is NOT flagged — that's
      "expensive but working," a reallocation question, not a defund one.)

    FREQUENCY CAP flag:
      A channel is a frequency-cap candidate if its impression volume is
      in the top quartile (within brand) but its Shapley share is below
      the brand median — spending a lot of impressions for comparatively
      little credit.
    """
    cpa_df = cpa_df.copy()

    min_cpa_per_brand = cpa_df.groupby(COL_BRAND)["cpa"].transform("min")
    cpa_df["cpa_vs_brand_min"] = cpa_df["cpa"] / min_cpa_per_brand
    cpa_df["defund_candidate"] = (
        (cpa_df["cpa_vs_brand_min"] >= defund_multiplier) &
        (cpa_df["shapley_pct"] < defund_credit_threshold)
    )

    impr_pctile = cpa_df.groupby(COL_BRAND)["n_impressions"].rank(pct=True)
    median_shapley_per_brand = cpa_df.groupby(COL_BRAND)["shapley_pct"].transform("median")
    cpa_df["frequency_cap_candidate"] = (
        (impr_pctile >= high_impression_threshold_pctile) &
        (cpa_df["shapley_pct"] < median_shapley_per_brand)
    )

    return cpa_df

In [19]:
# ─────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    per_brand_attribution = pd.read_csv("attribution_per_brand.csv")
    touchpoints_df = pd.read_csv("touchpoints_clean_v3.csv")
    spend_df = pd.read_csv("campaign_spend.csv")

    print("=" * 70)
    print("STEP A: Deriving ACTUAL spend per brand-channel from real event")
    print("        counts x Cost_Rate_INR (CPC/CPM-aware)")
    print("=" * 70)
    spend_actual = compute_actual_spend(touchpoints_df, spend_df)
    print(spend_actual.to_string(index=False))
    spend_actual.to_csv("spend_actual.csv", index=False)

    print("\n" + "=" * 70)
    print("STEP B: CPA using Shapley-attributed conversions")
    print("=" * 70)
    cpa_df = compute_cpa(per_brand_attribution, spend_actual)
    cpa_df = flag_defund_and_frequency_cap(cpa_df)
    cols = [COL_BRAND, COL_CH, "shapley_pct", "attributed_conversions",
            "actual_spend", "cpa", "pricing_model",
            "defund_candidate", "frequency_cap_candidate"]
    print(cpa_df[cols].to_string(index=False))
    cpa_df.to_csv("cpa_final.csv", index=False)

    print("\n" + "=" * 70)
    print("DEFUND CANDIDATES (high CPA, low credit)")
    print("=" * 70)
    print(cpa_df[cpa_df["defund_candidate"]][[COL_BRAND, COL_CH, "cpa", "shapley_pct"]]
          .to_string(index=False))

    print("\n" + "=" * 70)
    print("FREQUENCY CAP CANDIDATES (high impression volume, low credit)")
    print("=" * 70)
    print(cpa_df[cpa_df["frequency_cap_candidate"]][[COL_BRAND, COL_CH, "n_impressions", "shapley_pct"]]
          .to_string(index=False))

    print("\nSaved: spend_actual.csv")
    print("Saved: cpa_final.csv")


STEP A: Deriving ACTUAL spend per brand-channel from real event
        counts x Cost_Rate_INR (CPC/CPM-aware)
brand_id         channel pricing_model  cost_rate_inr  budget_allocated  n_impressions  n_clicks  actual_spend
     B01       Instagram           CPM         391.98         613144.27           6907       433       2707.41
     B01   Google Search           CPC          23.41       27896859.25           6911       448      10487.68
     B01 Influencer Blog           CPM         329.34       14499430.43           6832       443       2250.05
     B01         Youtube           CPM         180.36       19807806.26           6823       419       1230.60
     B01     Marketplace           CPC          37.54       37182759.79           6855       401      15053.54
     B02       Instagram           CPM         212.40       21083427.60           6810       421       1446.44
     B02   Google Search           CPC          43.14       23670390.50           6865       421      18161.94
 